# Modul 19: Sequenzmodelle, Autoencoder und Generierung

    **Notebooktyp:** Übungen mit ausführlichen Lösungen  
    **Vorlesungen dieses Moduls:** Sequenzmodelle, Autoencoder und Generierung  
    **Erwarteter Schwierigkeitsgrad:** Fortgeschrittene Keras-Anwendung mit zeitlichen Daten und Rekonstruktion  
    **Orientierungszeit:** etwa 170 bis 240 Minuten

    ## Überblick

    Sie erstellen zeitlich korrekte Sequenzfenster, vergleichen naive Modelle mit Conv1D, SimpleRNN und GRU und analysieren zeitabhängige Fehler. Danach trainieren Sie dichte Autoencoder für Rekonstruktion, Denoising, Anomaliehinweise und latente Interpolation.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_19A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_19B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - Sequenzen in Batch-, Zeit- und Merkmalsachsen strukturieren.
- Zeitliche Splits ohne Zukunftsleckage erstellen und naive Baselines berechnen.
- Conv1D, SimpleRNN und GRU unter gleichen Bedingungen vergleichen.
- Maskierte beziehungsweise gepaddete Sequenzen korrekt behandeln.
- Fehler zeitlich und nach Betriebsabschnitten analysieren.
- Encoder, Decoder, latente Darstellung und Rekonstruktionsverlust praktisch umsetzen.
- Denoising, Anomalieerkennung und latente Interpolation vorsichtig interpretieren.

    ## Bewertete Fähigkeiten

    - Sequenzfenster, zeitliche Splits und naive Baselines
- Conv1D, SimpleRNN und GRU in Keras
- Masking, Padding und zeitliche Fehleranalyse
- dichte Autoencoder, latente Codes und Rekonstruktionsverlust
- Denoising, Anomalieschwellen und latente Interpolation

## Arbeitsanweisungen

Dieses Lösungsnotebook entspricht dem Übungsnotebook Aufgabe für Aufgabe. Führen Sie es von oben nach unten aus und vergleichen Sie nicht nur Endwerte, sondern auch Vorgehen, Formprüfungen, Datenaufteilung und Interpretation. Die Kommentare erklären bewusst auch typische Fehlerquellen und methodische Entscheidungen.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# TensorFlow ist in Google Colab üblicherweise bereits verfügbar.
# Der Fallback installiert nur dann eine CPU-Version, wenn der Import fehlt.
import os
import sys
import subprocess
import warnings
from pathlib import Path

try:
    import tensorflow as tf
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "tensorflow-cpu"])
    import tensorflow as tf

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_SEED = 42
FAST_MODE = os.environ.get("COURSE_FAST", "0") == "1"
OFFLINE_MODE = os.environ.get("COURSE_OFFLINE", "0") == "1"

np.random.seed(RANDOM_SEED)
tf.keras.utils.set_random_seed(RANDOM_SEED)
warnings.filterwarnings("ignore", category=FutureWarning)

from sklearn.datasets import load_digits
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

# Eine lokale synthetische Messreihe kombiniert mehrere Frequenzen,
# einen langsamen Trend und reproduzierbares Rauschen.
sequence_rng_19 = np.random.default_rng(RANDOM_SEED)
time_index_19 = np.arange(0, 1800, dtype=np.float32)
raw_signal_19 = (
    np.sin(time_index_19 / 18.0)
    + 0.35 * np.sin(time_index_19 / 5.5)
    + 0.0007 * time_index_19
    + sequence_rng_19.normal(0.0, 0.08, size=len(time_index_19))
).astype("float32")

digits_19 = load_digits()
digit_images_19 = (digits_19.images.astype("float32") / 16.0)
digit_vectors_19 = digit_images_19.reshape(len(digit_images_19), -1)
digit_labels_19 = digits_19.target.astype("int64")
AE_train_valid_19, AE_test_19, AE_y_train_valid_19, AE_y_test_19 = train_test_split(
    digit_vectors_19,
    digit_labels_19,
    test_size=0.20,
    stratify=digit_labels_19,
    random_state=RANDOM_SEED,
)
AE_train_19, AE_valid_19, AE_y_train_19, AE_y_valid_19 = train_test_split(
    AE_train_valid_19,
    AE_y_train_valid_19,
    test_size=0.25,
    stratify=AE_y_train_valid_19,
    random_state=RANDOM_SEED,
)

print("Rohsequenz:", raw_signal_19.shape)
print("Autoencoder Train/Valid/Test:", AE_train_19.shape, AE_valid_19.shape, AE_test_19.shape)

print("TensorFlow-Version:", tf.__version__)
print("Schneller Validierungsmodus:", FAST_MODE)


## Aufgabe 1: Sequenzfenster, zeitliche Splits und Baselines

    Bereiten Sie die synthetische Messreihe als Vorhersageproblem vor.

1. Schreiben Sie eine Funktion, die aus einer eindimensionalen Reihe Fenster der Länge 24 und jeweils den direkt folgenden Zielwert erzeugt.
2. Bringen Sie die Merkmale in die Form `(Beispiele, Zeitschritte, Merkmale)`.
3. Teilen Sie chronologisch in 60 Prozent Training, 20 Prozent Validierung und 20 Prozent Test. Verwenden Sie kein zufälliges Shuffling vor dem Split.
4. Berechnen Sie eine naive Persistenzbaseline, die den letzten Fensterwert vorhersagt.
5. Trainieren Sie zusätzlich Ridge Regression auf den abgeflachten Trainingsfenstern.
6. Vergleichen Sie Validierungs-MAE und Test-MAE beider Baselines und visualisieren Sie einen Ausschnitt der Testvorhersagen.

> **Hinweis:** Das Ziel eines Fensters beginnt genau an der Position direkt hinter dem letzten Eingabewert.

In [ ]:
window_length_19 = 24

# Speichern Sie die resultierenden Partitionen als X_seq_train_19,
# X_seq_valid_19, X_seq_test_19 und entsprechende y-Variablen.

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Sequenzfenster, zeitliche Splits und Baselines
#
# Ziel dieser Codezelle:
# Bereiten Sie die synthetische Messreihe als Vorhersageproblem vor. 1. Schreiben
# Sie eine Funktion, die aus einer eindimensionalen Reihe Fenster der Länge 24 und
# jeweils den direkt folgenden Zielwert erzeugt. 2. Bringe...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

window_length_19 = 24

def make_sequence_windows_19(series, window_length):
    series = np.asarray(series, dtype=np.float32)
    if series.ndim != 1:
        raise ValueError("Die Eingabereihe muss eindimensional sein.")
    if window_length < 1 or window_length >= len(series):
        raise ValueError("Die Fensterlänge ist unzulässig.")

    windows = []
    targets = []
    for start in range(len(series) - window_length):
        stop = start + window_length
        windows.append(series[start:stop])
        targets.append(series[stop])
    return np.asarray(windows), np.asarray(targets)

X_sequence_19, y_sequence_19 = make_sequence_windows_19(
    raw_signal_19,
    window_length_19,
)
# Die letzte Achse ist die Merkmalsachse. Hier gibt es pro Zeitpunkt
# genau einen Messwert.
X_sequence_19 = X_sequence_19[..., np.newaxis]

number_of_examples_19 = len(X_sequence_19)
train_end_19 = int(0.60 * number_of_examples_19)
valid_end_19 = int(0.80 * number_of_examples_19)

X_seq_train_19 = X_sequence_19[:train_end_19]
y_seq_train_19 = y_sequence_19[:train_end_19]
X_seq_valid_19 = X_sequence_19[train_end_19:valid_end_19]
y_seq_valid_19 = y_sequence_19[train_end_19:valid_end_19]
X_seq_test_19 = X_sequence_19[valid_end_19:]
y_seq_test_19 = y_sequence_19[valid_end_19:]

assert X_seq_train_19.shape[1:] == (window_length_19, 1)
assert len(X_seq_train_19) + len(X_seq_valid_19) + len(X_seq_test_19) == number_of_examples_19

# Persistenz nutzt ausschließlich den letzten bekannten Messwert.
persistence_valid_19 = X_seq_valid_19[:, -1, 0]
persistence_test_19 = X_seq_test_19[:, -1, 0]

# Ridge erhält dieselben Zeitfenster, aber als zweidimensionale Tabelle.
ridge_sequence_19 = Ridge(alpha=1.0)
ridge_sequence_19.fit(
    X_seq_train_19.reshape(len(X_seq_train_19), -1),
    y_seq_train_19,
)
ridge_valid_19 = ridge_sequence_19.predict(
    X_seq_valid_19.reshape(len(X_seq_valid_19), -1)
)
ridge_test_19 = ridge_sequence_19.predict(
    X_seq_test_19.reshape(len(X_seq_test_19), -1)
)

baseline_comparison_19 = pd.DataFrame(
    {
        "model": ["Persistenz", "Ridge"],
        "validation_mae": [
            mean_absolute_error(y_seq_valid_19, persistence_valid_19),
            mean_absolute_error(y_seq_valid_19, ridge_valid_19),
        ],
        "test_mae": [
            mean_absolute_error(y_seq_test_19, persistence_test_19),
            mean_absolute_error(y_seq_test_19, ridge_test_19),
        ],
    }
)
print("Fensterform:", X_sequence_19.shape)
print(baseline_comparison_19.round(4).to_string(index=False))

preview_count_19 = 140
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(y_seq_test_19[:preview_count_19], label="wahr")
ax.plot(persistence_test_19[:preview_count_19], label="Persistenz")
ax.plot(ridge_test_19[:preview_count_19], label="Ridge")
ax.set_title("Baselines auf einem zeitlichen Testausschnitt")
ax.set_xlabel("Testzeitpunkt")
ax.set_ylabel("Signalwert")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 1

Ein zeitlicher Split simuliert die Vorhersage zukünftiger Werte aus früheren Daten. Ein zufälliger Split würde benachbarte und stark ähnliche Fenster auf beide Seiten verteilen und könnte Zukunftsinformation indirekt zugänglich machen. Die Persistenzbaseline ist für glatte Signale oft überraschend stark. Ein neuronales Modell sollte deshalb nicht nur einen niedrigen Fehler, sondern einen nachvollziehbaren Vorteil gegenüber einfachen Baselines zeigen.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 2: Conv1D, SimpleRNN und GRU fair vergleichen

    Trainieren Sie drei kleine Sequenzmodelle unter gleichen Bedingungen.

1. Erstellen Sie je ein Modell mit `Conv1D`, `SimpleRNN` und `GRU`.
2. Halten Sie Eingabeform, Ausgabeschicht, Optimierer, Loss, maximale Epochen und Batchgröße gleichartig.
3. Verwenden Sie MAE als Loss und Metrik sowie Early Stopping auf `val_loss`.
4. Speichern Sie Trainingsdauer, beste Validierungs-MAE, Test-MAE und Parameterzahl.
5. Wählen Sie das beste Modell ausschließlich anhand der Validierung und speichern Sie es als `best_sequence_model_19`.
6. Vergleichen Sie das ausgewählte Modell mit Persistenz und Ridge auf dem Testset.

> **Hinweis:** Erstellen Sie für jedes Modell eine neue Instanz und setzen Sie den Seed vor dem Aufbau.

In [ ]:
sequence_epochs_19 = 4 if FAST_MODE else 25
sequence_batch_size_19 = 32

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Conv1D, SimpleRNN und GRU fair vergleichen
#
# Ziel dieser Codezelle:
# Trainieren Sie drei kleine Sequenzmodelle unter gleichen Bedingungen. 1. Erstellen
# Sie je ein Modell mit Conv1D, SimpleRNN und GRU. 2. Halten Sie Eingabeform,
# Ausgabeschicht, Optimierer, Loss, maximale Epochen und Bat...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

sequence_epochs_19 = 4 if FAST_MODE else 25
sequence_batch_size_19 = 32

def build_sequence_model_19(model_type):
    inputs = tf.keras.Input(shape=(window_length_19, 1))
    if model_type == "Conv1D":
        x = tf.keras.layers.Conv1D(12, kernel_size=5, activation="relu")(inputs)
        x = tf.keras.layers.GlobalAveragePooling1D()(x)
    elif model_type == "SimpleRNN":
        x = tf.keras.layers.SimpleRNN(12)(inputs)
    elif model_type == "GRU":
        x = tf.keras.layers.GRU(12)(inputs)
    else:
        raise ValueError(f"Unbekannter Modelltyp: {model_type}")
    outputs = tf.keras.layers.Dense(1)(x)
    model = tf.keras.Model(inputs, outputs, name=f"sequence_{model_type.lower()}")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(0.002),
        loss="mae",
        metrics=["mae"],
    )
    return model

sequence_results_19 = []
sequence_models_19 = {}
sequence_histories_19 = {}

for model_index, model_type in enumerate(["Conv1D", "SimpleRNN", "GRU"]):
    tf.keras.utils.set_random_seed(RANDOM_SEED + model_index)
    model = build_sequence_model_19(model_type)
    start_time = time.perf_counter()
    history = model.fit(
        X_seq_train_19,
        y_seq_train_19,
        validation_data=(X_seq_valid_19, y_seq_valid_19),
        epochs=sequence_epochs_19,
        batch_size=sequence_batch_size_19,
        callbacks=[
            tf.keras.callbacks.EarlyStopping(
                monitor="val_loss",
                patience=4,
                restore_best_weights=True,
            )
        ],
        verbose=0,
    )
    training_seconds = time.perf_counter() - start_time
    test_predictions = model.predict(X_seq_test_19, verbose=0).ravel()
    test_mae = mean_absolute_error(y_seq_test_19, test_predictions)
    best_validation_mae = min(history.history["val_mae"])

    sequence_models_19[model_type] = model
    sequence_histories_19[model_type] = history.history
    sequence_results_19.append(
        {
            "model": model_type,
            "best_validation_mae": best_validation_mae,
            "test_mae": test_mae,
            "parameters": model.count_params(),
            "training_seconds": training_seconds,
        }
    )

sequence_comparison_19 = pd.DataFrame(sequence_results_19).sort_values(
    "best_validation_mae"
)
best_sequence_name_19 = sequence_comparison_19.iloc[0]["model"]
best_sequence_model_19 = sequence_models_19[best_sequence_name_19]
best_sequence_test_predictions_19 = best_sequence_model_19.predict(
    X_seq_test_19,
    verbose=0,
).ravel()

extended_comparison_19 = pd.concat(
    [
        baseline_comparison_19[["model", "validation_mae", "test_mae"]],
        sequence_comparison_19.rename(
            columns={"best_validation_mae": "validation_mae"}
        )[["model", "validation_mae", "test_mae"]],
    ],
    ignore_index=True,
).sort_values("validation_mae")

print("Ausgewähltes Sequenzmodell:", best_sequence_name_19)
print(sequence_comparison_19.round(4).to_string(index=False))
print("\nMit Baselines:")
print(extended_comparison_19.round(4).to_string(index=False))

### Reflexion zu Aufgabe 2

Conv1D erkennt lokale zeitliche Muster parallel, SimpleRNN verarbeitet den Zustand schrittweise, und GRU besitzt Gates zur kontrollierten Informationsweitergabe. Ein kleines Experiment beweist keine allgemeine Überlegenheit einer Architektur. Die Auswahl muss auf der Validierung erfolgen, während der Testwert nur der abschließenden Berichterstattung dient. Parameterzahl und Trainingszeit ergänzen die reine Fehlermetrik.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 3: Maskierte Sequenzen und zeitliche Fehleranalyse

    Untersuchen Sie variable Sequenzlängen und die Fehlerentwicklung über die Zeit.

1. Erstellen Sie mehrere Sequenzen unterschiedlicher Länge und padden Sie sie rechts mit dem Wert `-999.0`.
2. Bauen Sie ein kleines Modell aus `Masking`, `GRU` und Dense-Ausgabe.
3. Zeigen Sie mit zwei identischen gültigen Sequenzen und unterschiedlich viel Padding, dass maskierte Zusatzwerte die Ausgabe praktisch nicht verändern.
4. Berechnen Sie für das in Aufgabe 2 ausgewählte Sequenzmodell absolute Testfehler.
5. Fassen Sie die Fehler für vier aufeinanderfolgende zeitliche Testabschnitte zusammen.
6. Visualisieren Sie wahre Werte, Vorhersagen und absoluten Fehler über einen längeren Testausschnitt.

> **Hinweis:** Padding muss immer denselben speziellen Wert verwenden wie die Masking-Schicht.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Maskierte Sequenzen und zeitliche Fehleranalyse
#
# Ziel dieser Codezelle:
# Untersuchen Sie variable Sequenzlängen und die Fehlerentwicklung über die Zeit. 1.
# Erstellen Sie mehrere Sequenzen unterschiedlicher Länge und padden Sie sie rechts
# mit dem Wert -999.0. 2. Bauen Sie ein kleines Modell...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

variable_sequences_19 = [
    np.array([0.2, 0.4, 0.1, 0.3], dtype=np.float32),
    np.array([1.0, 0.8, 0.6], dtype=np.float32),
    np.array([-0.2, 0.0, 0.1, 0.2, 0.4], dtype=np.float32),
]
padded_sequences_19 = tf.keras.utils.pad_sequences(
    variable_sequences_19,
    padding="post",
    value=-999.0,
    dtype="float32",
)[..., np.newaxis]

tf.keras.utils.set_random_seed(RANDOM_SEED)
masked_inputs_19 = tf.keras.Input(shape=(None, 1))
masked_x_19 = tf.keras.layers.Masking(mask_value=-999.0)(masked_inputs_19)
masked_x_19 = tf.keras.layers.GRU(6)(masked_x_19)
masked_outputs_19 = tf.keras.layers.Dense(1)(masked_x_19)
masked_model_19 = tf.keras.Model(masked_inputs_19, masked_outputs_19)

# Beide Eingaben enthalten dieselben vier gültigen Werte. Die zweite
# besitzt nur zwei zusätzliche Paddingpositionen.
short_padded_19 = np.array([[[0.2], [0.4], [0.1], [0.3], [-999.0]]], dtype=np.float32)
long_padded_19 = np.array(
    [[[0.2], [0.4], [0.1], [0.3], [-999.0], [-999.0], [-999.0]]],
    dtype=np.float32,
)
short_output_19 = masked_model_19(short_padded_19, training=False).numpy()
long_output_19 = masked_model_19(long_padded_19, training=False).numpy()
np.testing.assert_allclose(short_output_19, long_output_19, atol=1e-6)

absolute_errors_19 = np.abs(
    y_seq_test_19 - best_sequence_test_predictions_19
)
temporal_groups_19 = np.array_split(np.arange(len(absolute_errors_19)), 4)
temporal_error_rows_19 = []
for group_number, indices in enumerate(temporal_groups_19, start=1):
    temporal_error_rows_19.append(
        {
            "test_section": group_number,
            "start_index": int(indices[0]),
            "end_index": int(indices[-1]),
            "mae": float(np.mean(absolute_errors_19[indices])),
        }
    )
temporal_error_table_19 = pd.DataFrame(temporal_error_rows_19)

print("Gepaddete Form:", padded_sequences_19.shape)
print("Ausgabe mit kurzem Padding:", short_output_19.ravel())
print("Ausgabe mit längerem Padding:", long_output_19.ravel())
print(temporal_error_table_19.round(4).to_string(index=False))

preview_count_19 = min(280, len(y_seq_test_19))
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(y_seq_test_19[:preview_count_19], label="wahr")
ax.plot(best_sequence_test_predictions_19[:preview_count_19], label="vorhergesagt")
ax.set_title(f"Zeitliche Vorhersagen: {best_sequence_name_19}")
ax.set_xlabel("Testzeitpunkt")
ax.set_ylabel("Signalwert")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(absolute_errors_19[:preview_count_19])
ax.set_title("Absoluter Fehler über die Testzeit")
ax.set_xlabel("Testzeitpunkt")
ax.set_ylabel("absoluter Fehler")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 3

Padding erzeugt künstliche Werte, die ohne Maske das Sequenzmodell beeinflussen würden. `Masking` teilt kompatiblen rekurrenten Schichten mit, welche Zeitschritte ignoriert werden sollen. Nicht jede Schicht unterstützt Masken, insbesondere eine gewöhnliche Conv1D-Pipeline behandelt Paddingwerte zunächst wie echte Eingaben. Zeitlich gruppierte Fehler können Drift, Regimewechsel oder wachsende Unsicherheit sichtbar machen, selbst wenn eine einzige Gesamt-MAE unauffällig wirkt.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 4: Einen dichten Autoencoder trainieren und den latenten Raum untersuchen

    Trainieren Sie einen Autoencoder auf den kleinen Ziffernbildern.

1. Erstellen Sie einen Encoder `64 -> 32 -> 12` und einen Decoder `12 -> 32 -> 64`.
2. Verwenden Sie Sigmoid in der Rekonstruktionsausgabe und MSE als Loss.
3. Trainieren Sie mit Eingabe gleich Ziel und Early Stopping.
4. Berechnen Sie den Rekonstruktions-MSE auf Training, Validierung und Test.
5. Visualisieren Sie sechs Originale und Rekonstruktionen.
6. Erzeugen Sie latente Codes für das Testset und stellen Sie die ersten beiden latenten Dimensionen nach Ziffernklasse dar. Interpretieren Sie diese Projektion vorsichtig.

> **Hinweis:** Eingabe und Ziel sind beim klassischen Autoencoder identisch.

In [ ]:
autoencoder_epochs_19 = 6 if FAST_MODE else 35
latent_dimension_19 = 12

# Speichern Sie die Modelle als encoder_19, decoder_19 und autoencoder_19.

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Einen dichten Autoencoder trainieren und den latenten Raum untersuchen
#
# Ziel dieser Codezelle:
# Trainieren Sie einen Autoencoder auf den kleinen Ziffernbildern. 1. Erstellen Sie
# einen Encoder 64 - 32 - 12 und einen Decoder 12 - 32 - 64. 2. Verwenden Sie
# Sigmoid in der Rekonstruktionsausgabe und MSE als Loss. 3....
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

autoencoder_epochs_19 = 6 if FAST_MODE else 35
latent_dimension_19 = 12
tf.keras.utils.set_random_seed(RANDOM_SEED)

encoder_inputs_19 = tf.keras.Input(shape=(64,), name="digit_vector")
encoder_hidden_19 = tf.keras.layers.Dense(32, activation="relu")(encoder_inputs_19)
latent_codes_19 = tf.keras.layers.Dense(
    latent_dimension_19,
    activation="linear",
    name="latent_code",
)(encoder_hidden_19)
encoder_19 = tf.keras.Model(encoder_inputs_19, latent_codes_19, name="encoder")

decoder_inputs_19 = tf.keras.Input(shape=(latent_dimension_19,), name="latent_input")
decoder_hidden_19 = tf.keras.layers.Dense(32, activation="relu")(decoder_inputs_19)
decoder_outputs_19 = tf.keras.layers.Dense(64, activation="sigmoid")(decoder_hidden_19)
decoder_19 = tf.keras.Model(decoder_inputs_19, decoder_outputs_19, name="decoder")

autoencoder_inputs_19 = tf.keras.Input(shape=(64,))
autoencoder_outputs_19 = decoder_19(encoder_19(autoencoder_inputs_19))
autoencoder_19 = tf.keras.Model(
    autoencoder_inputs_19,
    autoencoder_outputs_19,
    name="dense_autoencoder",
)
autoencoder_19.compile(
    optimizer=tf.keras.optimizers.Adam(0.001),
    loss="mse",
)

autoencoder_history_19 = autoencoder_19.fit(
    AE_train_19,
    AE_train_19,
    validation_data=(AE_valid_19, AE_valid_19),
    epochs=autoencoder_epochs_19,
    batch_size=32,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
        )
    ],
    verbose=0,
)

def reconstruction_mse_19(model, values):
    reconstructions = model.predict(values, verbose=0)
    return float(np.mean((values - reconstructions) ** 2)), reconstructions

train_reconstruction_mse_19, _ = reconstruction_mse_19(autoencoder_19, AE_train_19)
valid_reconstruction_mse_19, valid_reconstructions_19 = reconstruction_mse_19(
    autoencoder_19,
    AE_valid_19,
)
test_reconstruction_mse_19, test_reconstructions_19 = reconstruction_mse_19(
    autoencoder_19,
    AE_test_19,
)

print("Train-MSE:", round(train_reconstruction_mse_19, 5))
print("Validierungs-MSE:", round(valid_reconstruction_mse_19, 5))
print("Test-MSE:", round(test_reconstruction_mse_19, 5))

example_indices_19 = np.arange(6)
fig, axes = plt.subplots(2, 6, figsize=(10, 4))
for column, index in enumerate(example_indices_19):
    axes[0, column].imshow(AE_test_19[index].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    axes[0, column].set_title(f"Original {AE_y_test_19[index]}")
    axes[1, column].imshow(test_reconstructions_19[index].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    axes[1, column].set_title("Rekonstruktion")
    axes[0, column].axis("off")
    axes[1, column].axis("off")
plt.tight_layout()
plt.show()

test_latent_codes_19 = encoder_19.predict(AE_test_19, verbose=0)
fig, ax = plt.subplots(figsize=(7, 5))
scatter = ax.scatter(
    test_latent_codes_19[:, 0],
    test_latent_codes_19[:, 1],
    c=AE_y_test_19,
    alpha=0.65,
)
ax.set_title("Erste zwei Dimensionen des 12-dimensionalen latenten Raums")
ax.set_xlabel("latente Dimension 1")
ax.set_ylabel("latente Dimension 2")
fig.colorbar(scatter, ax=ax, label="Ziffer")
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 4

Der Autoencoder lernt ohne Klassenlabels eine komprimierte Darstellung, aus der er die Eingabe rekonstruiert. Niedriger Rekonstruktionsfehler bedeutet Ähnlichkeit auf der gewählten Pixelmetrik, nicht automatisch semantisches Verständnis. Eine Darstellung nur der ersten zwei von zwölf latenten Dimensionen kann Klassenmuster verdecken oder zufällig hervorheben. Eine saubere Bewertung verwendet ungesehene Testbilder und vergleicht den Fehler mit einfachen linearen Alternativen, wenn die Modellwahl davon abhängt.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Aufgabe 5: Integration: Denoising, Anomaliehinweise und latente Interpolation

    Erweitern Sie den Autoencoder-Workflow.

1. Fügen Sie den Trainingsbildern reproduzierbares Gaußrauschen hinzu und begrenzen Sie Werte auf `[0, 1]`.
2. Trainieren Sie einen neuen Autoencoder, der verrauschte Eingaben auf saubere Ziele abbildet.
3. Visualisieren Sie verrauschte Eingaben, saubere Bilder und Denoising-Rekonstruktionen.
4. Bestimmen Sie eine Anomalieschwelle als 95. Perzentil der Rekonstruktionsfehler auf **sauberen Validierungsdaten**.
5. Erzeugen Sie künstliche Anomalien durch zufälliges Mischen der Pixel einiger Testbilder und prüfen Sie, welcher Anteil oberhalb der Schwelle liegt.
6. Interpolieren Sie im latenten Raum zwischen zwei Testziffern und decodieren Sie fünf Zwischenpunkte.
7. Erläutern Sie Grenzen einer solchen Anomalieentscheidung und einer Interpretation interpolierter Bilder als echte Datengenerierung.

> **Hinweis:** Bestimmen Sie die Schwelle ohne Testanomalien, sonst wird die Bewertung verzerrt.

In [ ]:
denoising_epochs_19 = 6 if FAST_MODE else 35

# ============================================================


In [ ]:
# ============================================================
# KOMMENTIERTE MUSTERLÖSUNG: Integration: Denoising, Anomaliehinweise und latente Interpolation
#
# Ziel dieser Codezelle:
# Erweitern Sie den Autoencoder-Workflow. 1. Fügen Sie den Trainingsbildern
# reproduzierbares Gaußrauschen hinzu und begrenzen Sie Werte auf [0, 1]. 2.
# Trainieren Sie einen neuen Autoencoder, der verrauschte Eingaben auf...
#
# Die Lösung folgt bewusst einer gut prüfbaren Schrittfolge.
# Zwischenwerte und Ausgaben machen Formen, Annahmen und Ergebnisse sichtbar.
# Die fachliche Interpretation und typische Fehlerquellen stehen in der
# ausführlichen Markdown-Reflexion direkt unter dieser Codezelle.
# ============================================================

denoising_epochs_19 = 6 if FAST_MODE else 35
denoising_rng_19 = np.random.default_rng(RANDOM_SEED + 5)

def add_noise_19(clean_values, standard_deviation=0.25):
    noise = denoising_rng_19.normal(
        0.0,
        standard_deviation,
        size=clean_values.shape,
    ).astype(np.float32)
    return np.clip(clean_values + noise, 0.0, 1.0)

noisy_train_19 = add_noise_19(AE_train_19)
noisy_valid_19 = add_noise_19(AE_valid_19)
noisy_test_19 = add_noise_19(AE_test_19)

tf.keras.utils.set_random_seed(RANDOM_SEED + 5)
denoising_input_19 = tf.keras.Input(shape=(64,))
denoising_hidden_19 = tf.keras.layers.Dense(32, activation="relu")(denoising_input_19)
denoising_latent_19 = tf.keras.layers.Dense(12, name="denoising_latent")(denoising_hidden_19)
denoising_encoder_19 = tf.keras.Model(
    denoising_input_19,
    denoising_latent_19,
    name="denoising_encoder",
)

denoising_decoder_input_19 = tf.keras.Input(shape=(12,))
denoising_decoder_hidden_19 = tf.keras.layers.Dense(32, activation="relu")(
    denoising_decoder_input_19
)
denoising_decoder_output_19 = tf.keras.layers.Dense(64, activation="sigmoid")(
    denoising_decoder_hidden_19
)
denoising_decoder_19 = tf.keras.Model(
    denoising_decoder_input_19,
    denoising_decoder_output_19,
    name="denoising_decoder",
)

denoising_output_19 = denoising_decoder_19(denoising_encoder_19(denoising_input_19))
denoising_autoencoder_19 = tf.keras.Model(
    denoising_input_19,
    denoising_output_19,
    name="denoising_autoencoder",
)
denoising_autoencoder_19.compile(optimizer="adam", loss="mse")
denoising_autoencoder_19.fit(
    noisy_train_19,
    AE_train_19,
    validation_data=(noisy_valid_19, AE_valid_19),
    epochs=denoising_epochs_19,
    batch_size=32,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True,
        )
    ],
    verbose=0,
)

denoised_test_19 = denoising_autoencoder_19.predict(noisy_test_19, verbose=0)
fig, axes = plt.subplots(3, 5, figsize=(9, 5.5))
for column in range(5):
    axes[0, column].imshow(noisy_test_19[column].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    axes[1, column].imshow(AE_test_19[column].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    axes[2, column].imshow(denoised_test_19[column].reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    for row in range(3):
        axes[row, column].axis("off")
axes[0, 0].set_ylabel("verrauscht")
axes[1, 0].set_ylabel("sauber")
axes[2, 0].set_ylabel("rekonstruiert")
plt.tight_layout()
plt.show()

# Die Schwelle wird nur aus sauberen Validierungsdaten abgeleitet.
clean_valid_reconstructions_19 = denoising_autoencoder_19.predict(AE_valid_19, verbose=0)
clean_valid_errors_19 = np.mean(
    (AE_valid_19 - clean_valid_reconstructions_19) ** 2,
    axis=1,
)
anomaly_threshold_19 = float(np.quantile(clean_valid_errors_19, 0.95))

clean_test_subset_19 = AE_test_19[:80]
anomaly_subset_19 = AE_test_19[80:120].copy()
# Jedes Anomaliebild behält seine Pixelwerte, verliert aber die
# räumliche Ziffernstruktur durch eine unabhängige Permutation.
for row in range(len(anomaly_subset_19)):
    anomaly_subset_19[row] = anomaly_subset_19[row][
        denoising_rng_19.permutation(anomaly_subset_19.shape[1])
    ]

evaluation_values_19 = np.vstack([clean_test_subset_19, anomaly_subset_19])
evaluation_labels_19 = np.r_[
    np.zeros(len(clean_test_subset_19), dtype=int),
    np.ones(len(anomaly_subset_19), dtype=int),
]
evaluation_reconstructions_19 = denoising_autoencoder_19.predict(
    evaluation_values_19,
    verbose=0,
)
evaluation_errors_19 = np.mean(
    (evaluation_values_19 - evaluation_reconstructions_19) ** 2,
    axis=1,
)
anomaly_predictions_19 = (evaluation_errors_19 > anomaly_threshold_19).astype(int)
clean_flag_rate_19 = anomaly_predictions_19[evaluation_labels_19 == 0].mean()
anomaly_detection_rate_19 = anomaly_predictions_19[evaluation_labels_19 == 1].mean()

print("Validierungsbasierte Schwelle:", round(anomaly_threshold_19, 5))
print("Anteil markierter sauberer Testbilder:", round(float(clean_flag_rate_19), 3))
print("Anteil erkannter künstlicher Anomalien:", round(float(anomaly_detection_rate_19), 3))

# Latente Interpolation verbindet zwei Codes linear. Der Decoder zeigt,
# welche Bilder entlang dieses mathematischen Pfades entstehen.
first_index_19 = int(np.flatnonzero(AE_y_test_19 == 1)[0])
second_index_19 = int(np.flatnonzero(AE_y_test_19 == 7)[0])
endpoint_values_19 = AE_test_19[[first_index_19, second_index_19]]
endpoint_codes_19 = denoising_encoder_19.predict(endpoint_values_19, verbose=0)
interpolation_weights_19 = np.linspace(0.0, 1.0, 5, dtype=np.float32)
interpolated_codes_19 = np.array(
    [
        (1.0 - weight) * endpoint_codes_19[0] + weight * endpoint_codes_19[1]
        for weight in interpolation_weights_19
    ]
)
interpolated_images_19 = denoising_decoder_19.predict(
    interpolated_codes_19,
    verbose=0,
)

fig, axes = plt.subplots(1, 5, figsize=(9, 2.2))
for axis, weight, image in zip(
    axes,
    interpolation_weights_19,
    interpolated_images_19,
):
    axis.imshow(image.reshape(8, 8), cmap="gray", vmin=0, vmax=1)
    axis.set_title(f"α={weight:.2f}")
    axis.axis("off")
plt.tight_layout()
plt.show()

### Reflexion zu Aufgabe 5

Ein hoher Rekonstruktionsfehler ist nur ein Anomaliehinweis relativ zu Trainingsverteilung, Architektur, Loss und gewählter Schwelle. Manche echten Anomalien können leicht rekonstruiert werden, während ungewöhnliche, aber gültige Beispiele fälschlich markiert werden. Die 95-Prozent-Schwelle legt ungefähr fünf Prozent Fehlalarme auf den verwendeten sauberen Validierungsdaten nahe, nicht in jeder zukünftigen Population. Latente Interpolation erzeugt Decoder-Ausgaben zwischen zwei Codes, garantiert aber weder realistische noch statistisch korrekt gezogene neue Daten wie ein speziell trainiertes generatives Modell.

**Kontrollfrage:** Welche Annahme, Formprüfung oder Trennungsentscheidung war für die Korrektheit dieser Lösung besonders wichtig?

## Abschluss und Selbstkontrolle

Prüfen Sie nach dem Durcharbeiten, ob Sie jede Lösung ohne bloßes Kopieren erklären könnten. Achten Sie besonders auf die Stellen, an denen Datenleckage, unpassende Formen, falsche Metriken oder unkontrollierte Zufälligkeit zu scheinbar guten, aber methodisch falschen Ergebnissen führen könnten.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.